# Draft: Map-Reduce Summarization on Two Hearings

| Item | Description |
|---|---|
| Goal | Show that a local model can summarize a congressional hearing end to end, and show what the output looks like, before anything runs at scale. The median hearing in the sample is 36,505 words, about 49,000 tokens, the 90th percentile is 66,486 words, and the longest 376,848. No small model reads that in one pass, so the transcript is divided into chunks, each chunk is summarized, and the pieces are combined. Two hearings are enough to exercise both paths: one of median length, and one long enough to force the collapse step. |
| Input | `hearings_sst.parquet` from D1.4. The model from the Hugging Face Hub, Qwen3.5-4B by default; a token in `.env` is needed only for the gated Llama models. |
| Output | None written. The summaries are printed for reading. Step 14 reads the one-hearing length test under `Interim/lentest`. |
| Run | Last run 2026-08-20 on one NVIDIA A40; the cell outputs shown are from that run. Not rerun: a GPU step, and a draft. Step 14 was executed on 2026-10-31 and reads saved files only. |

## Step 1. Import packages

In [ ]:
import hashlib
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

## Step 2. Define file locations

Model weights go to the project cache on scratch, never to the home folder.

In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[1]  # <section>/<subfolder>/ sits two levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T1 - Text Summarization/T1.1 - Process Prototyping and Evaluation")
NOTEBOOK_NAME = "T1.1.1 - Draft - Map-Reduce on Two Hearings"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
INPUT_FOLDER = SCRATCH_FOLDER / "D1 - Data Collection and Processing/D1.4 - SST Sample Construction" / "Processed"
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T1 - Text Summarization" / ".env"  # HF_TOKEN for the gated models

MODEL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE)

## Step 3. Choose the model and the chunk budget

The model is the highest-impact choice in this stage, so it is a parameter, and the draft makes swapping it cheap. Four candidates fit in bfloat16 on a 48 GB card: Qwen3.5-4B (9.3 GB), gemma-4-E4B-it (16.0 GB), Llama-3.1-8B-Instruct (16.1 GB), and Llama-3.2-3B-Instruct (6.4 GB). The chunk budget of 6,144 tokens sits well inside every context window. A median hearing then becomes a handful of chunks, so a position is rarely separated from its speaker.

In [1]:
MODEL = os.environ.get("MODEL", "Qwen/Qwen3.5-4B")
CHUNK_TOKENS = 6144  # tokens per chunk, net of the instructions and the generated summary

# A token is needed only for gated models (the two Llama checkpoints). It is read from .env, never written into the notebook.
if ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():
        name, _, value = line.strip().partition("=")
        if name == "HF_TOKEN" and value:
            os.environ["HF_TOKEN"] = value.strip().strip('"').strip("'")

print(f"model      : {MODEL}")
print(f"cache      : {MODEL_CACHE}")
print(f"chunk      : {CHUNK_TOKENS:,} tokens")
print(f"gpu        : {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
print(f"hf token   : {'set' if os.environ.get('HF_TOKEN') else 'not set'}")

model      : Qwen/Qwen3.5-4B
cache      : /scratch2/acroh/projects/llm_method/hf_cache
chunk      : 6,144 tokens
gpu        : NVIDIA L40S, 48 GB
hf token   : set


## Step 4. Pick two representative hearings

One House hearing at the median length, which should summarize in one map pass and one reduce. One Senate hearing at the 90th percentile, long enough that the segment summaries may overflow and force the collapse step.

In [2]:
sst = pd.read_parquet(INPUT_FOLDER / "hearings_sst.parquet")
print(f"SST sample: {len(sst):,} hearings")
print(f"  words per hearing: median {sst['n_words'].median():,.0f}, p90 {sst['n_words'].quantile(0.9):,.0f}, max {sst['n_words'].max():,}")

house = sst.loc[sst["committee_scope"] == "house_science"]
senate = sst.loc[sst["committee_scope"] == "senate_commerce"]


def nearest(frame, quantile):
    target = frame["n_words"].quantile(quantile)
    return frame.iloc[(frame["n_words"] - target).abs().argsort().iloc[0]]


picks = [nearest(house, 0.50), nearest(senate, 0.90)]

for p, label in zip(picks, ("House, median length", "Senate, 90th percentile")):
    print(f"\n{label}")
    print(f"  {p['packageId']}  ({p['year']})")
    print(f"  {p['title'][:96]}")
    print(f"  {p['n_words']:,} words, {p['n_turns']:,} turns, {p['n_members']} members, {p['n_witnesses']} witnesses")

SST sample: 2,352 hearings
  words per hearing: median 36,505, p90 66,486, max 376,848

House, median length
  CHRG-112hhrg74059  (2012)
  HOW THE REPORT ON CARCINOGENS USES SCIENCE TO MEET ITS STATUTORY OBLIGATIONS, AND ITS IMPACT ON 
  22,896 words, 265 turns, 43 members, 8 witnesses

Senate, 90th percentile
  CHRG-110shrg74104  (2007)
  700 MHz AUCTION: PUBLIC SAFETY AND COMPETITION ISSUES
  72,757 words, 313 turns, 22 members, 7 witnesses


## Step 5. Load the model

`dtype="auto"` takes the precision the model was published in, bfloat16 for all four candidates. No quantization: none is needed on a 48 GB card, and it would be a change to the measurement.

In [3]:
started = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype="auto", device_map="cuda")
model.eval()

print(f"loaded in {time.time() - started:.0f}s")
print(f"  parameters   : {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"  dtype        : {next(model.parameters()).dtype}")
print(f"  GPU allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print(f"  context      : {getattr(model.config, 'max_position_embeddings', 'n/a'):,}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

loaded in 4s
  parameters   : 4.21B
  dtype        : torch.bfloat16
  GPU allocated: 8.4 GB
  context      : 262,144


## Step 6. Define the three prompts

Three jobs, three prompts. The map prompt reads transcript and reports what each speaker said. The collapse prompt condenses a run of segment summaries without concluding, since it sees only a middle portion. The reduce prompt is told its input is already summary and asks what the hearing was about.

In [ ]:
# One prompt for all three stages fails in a specific way: the reduce step then treats summaries as speech and re-attributes positions.
PROMPT_VERSION = "hearing-mapreduce-1.0"  # stamped on every logged call, so a summary can be traced to the wording that produced it
REDUCE_MAX_TOKENS = 512  # the summaries are later embedded by a 512-token encoder, so a longer summary would be cut, not longer
REDUCE_TARGET_WORDS = "200 to 250"  # the model writes about 1.55 tokens per word, so this lands near 390 tokens with room to finish
GENERATION = {"do_sample": True, "temperature": 0.1}  # near-greedy sampling, kept from the earlier pipeline

MAP_SYSTEM = ("You are an expert in text summarization with a working knowledge of United "
              "States congressional hearings. You report only what the transcript states.")

MAP_USER = """Below is one consecutive segment of a congressional hearing transcript. Each speaking turn is labelled with the speaker.

Summarize this segment in about 150 words. Cover:
- the subjects raised, and what was said about them;
- the position each named speaker takes, attributed to that speaker;
- any numbers, dollar amounts, dates, and bill or act names, copied exactly as they appear.

If the segment is procedural or ceremonial and contains none of these, say so in one sentence.

--- SEGMENT ---
{text}
--- END OF SEGMENT ---

Write only the summary of the segment above."""

COLLAPSE_SYSTEM = ("You are an expert in text summarization with a working knowledge of United "
                   "States congressional hearings. You condense summaries without adding to them.")

COLLAPSE_USER = """Below are summaries of consecutive segments of one congressional hearing, in the order they occurred. They are already summaries, not transcript.

Condense them into one account of about 200 words that keeps the named speakers, their positions, and the numbers and legislation. This is a middle portion of the hearing, so do not write an overall conclusion about the hearing.

--- SEGMENT SUMMARIES ---
{text}
--- END ---

Write only the condensed account."""

REDUCE_SYSTEM = ("You are an expert in text summarization with a working knowledge of United "
                 "States congressional hearings. You write from the summaries given and add "
                 "nothing that is not in them.")

REDUCE_USER = """Below are summaries of every segment of one congressional hearing, in order. They are already summaries, not transcript.

Write a single summary of the hearing as a whole, in about {target_words} words. Cover:
- what the hearing was about;
- the main subjects, the most heavily discussed first;
- the position each named speaker took;
- any point where speakers took opposing positions, naming both sides;
- the key numbers and the legislation referred to, exactly as given.

Where the segment summaries do not establish one of these, leave it out rather than inferring it.

--- SEGMENT SUMMARIES ---
{text}
--- END ---

Write only the summary of the hearing, in about {target_words} words, as continuous prose."""

## Step 7. Define how speaking turns are packed into chunks

The unit is the speaking turn, one person's uninterrupted contribution, labeled with the speaker so a position can be attributed to a person. Whole turns are packed in order until the next one would overflow the budget. Only a turn longer than the whole budget is divided, at a paragraph boundary, and `split_turn` marks where that happened. Questions submitted for the record are left out, because they were not spoken at the hearing.

In [ ]:
def format_turn(turn):
    """One turn as the model sees it: a speaker label, then what was said. The chair has no surname, so the honorific alone is the label."""
    name = turn.get("surname") or ""
    honorific = turn.get("honorific") or ""
    who = f"{honorific} {name}".strip() or "Unidentified speaker"
    if turn.get("section") == "qfr":
        who = f"{who} (submitted for the record)"
    return f"{who}: {turn['text']}"


def encoded_length(tokenizer, text):
    return len(tokenizer.encode(text, add_special_tokens=False))


def token_slice(text, tokenizer, max_tokens):
    """Cut text into pieces that each re-encode within the budget. decode then encode is not the identity, so every piece is re-checked."""
    ids = tokenizer.encode(text, add_special_tokens=False)
    pieces = []
    start = 0
    while start < len(ids):
        take = min(max_tokens, len(ids) - start)
        piece = tokenizer.decode(ids[start:start + take])
        while take > 64 and encoded_length(tokenizer, piece) > max_tokens:
            take = int(take * 0.9)
            piece = tokenizer.decode(ids[start:start + take])
        pieces.append(piece)
        start += take
    return pieces


def split_long_turn(text, tokenizer, max_tokens):
    """Divide one over-long turn at paragraph boundaries, and a paragraph that is still too long by tokens."""
    join_cost = encoded_length(tokenizer, "\n\n")
    out = []
    buffer = []
    buffer_tokens = 0
    for paragraph in text.split("\n\n"):
        n = encoded_length(tokenizer, paragraph)
        if n > max_tokens:
            if buffer:
                out.append("\n\n".join(buffer))
                buffer, buffer_tokens = [], 0
            out.extend(token_slice(paragraph, tokenizer, max_tokens))
            continue
        cost = n + (join_cost if buffer else 0)
        if buffer_tokens + cost > max_tokens:
            out.append("\n\n".join(buffer))
            buffer, buffer_tokens = [], 0
            cost = n
        buffer.append(paragraph)
        buffer_tokens += cost
    if buffer:
        out.append("\n\n".join(buffer))
    # Re-encode every piece: separators cost tokens and the tokenizer re-merges across a cut.
    verified = []
    for piece in out:
        if encoded_length(tokenizer, piece) <= max_tokens:
            verified.append(piece)
        else:
            verified.extend(token_slice(piece, tokenizer, max_tokens))
    return verified


def chunk_turns(turns, tokenizer, max_tokens, spoken_only=True):
    """Group consecutive whole turns into chunks of at most max_tokens tokens. Only a turn longer than the budget is divided."""
    selected = [t for t in turns if not (spoken_only and t.get("section") == "qfr")]
    join_cost = encoded_length(tokenizer, "\n\n")  # the blank lines between turns are tokens too
    chunks = []
    current = []
    current_tokens = 0
    current_speakers = []

    def flush(split=False):
        nonlocal current, current_tokens, current_speakers
        if not current:
            return
        text = "\n\n".join(current)
        speakers = sorted(set(current_speakers))
        if encoded_length(tokenizer, text) > max_tokens:  # the emitted text is what counts, not the sum of its parts
            for piece in split_long_turn(text, tokenizer, max_tokens):
                chunks.append({"text": piece, "n_turns": len(current), "n_tokens": encoded_length(tokenizer, piece),
                               "speakers": speakers, "split_turn": True})
        else:
            chunks.append({"text": text, "n_turns": len(current), "n_tokens": encoded_length(tokenizer, text),
                           "speakers": speakers, "split_turn": split})
        current, current_tokens, current_speakers = [], 0, []

    for turn in selected:
        rendered = format_turn(turn)
        n = len(tokenizer.encode(rendered, add_special_tokens=False))
        if n > max_tokens:  # one turn that cannot fit: emit what is pending, then divide this turn
            flush()
            for piece in split_long_turn(rendered, tokenizer, max_tokens):
                chunks.append({"text": piece, "n_turns": 1, "n_tokens": encoded_length(tokenizer, piece),
                               "speakers": [turn.get("surname") or "Unidentified speaker"], "split_turn": True})
            continue
        cost = n + (join_cost if current else 0)
        if current_tokens + cost > max_tokens:
            flush()
            cost = n
        current.append(rendered)
        current_tokens += cost
        current_speakers.append(turn.get("surname") or "Unidentified speaker")
    flush()
    return chunks

## Step 8. Chunk both hearings

Two numbers to read: `n_chunks`, and how many chunks came from a split turn. Every chunk is measured on the text the model receives, not on the packer's arithmetic.

In [5]:
chunked = []
for p in picks:
    chunks = chunk_turns(p["turns"], tokenizer, CHUNK_TOKENS, spoken_only=True)
    chunked.append(chunks)
    split = sum(c["split_turn"] for c in chunks)
    sizes = [c["n_tokens"] for c in chunks]
    print(f"{p['packageId']}: {len(chunks)} chunks from {p['n_turns']:,} turns")
    print(f"  tokens per chunk : min {min(sizes):,}  median {sorted(sizes)[len(sizes) // 2]:,}  max {max(sizes):,}")
    print(f"  turns per chunk  : {[c['n_turns'] for c in chunks]}")
    print(f"  chunks from a split turn: {split}")
    over = [s for s in sizes if s > CHUNK_TOKENS]
    assert not over, f"{len(over)} chunks exceed the {CHUNK_TOKENS}-token budget"
    print()

print("first 400 characters of the first chunk, as the model receives it:")
print("-" * 78)
print(chunked[0][0]["text"][:400])

CHRG-112hhrg74059: 6 chunks from 265 turns
  tokens per chunk : min 3,889  median 6,056  max 6,129
  turns per chunk  : [10, 55, 77, 18, 81, 24]
  chunks from a split turn: 0



CHRG-110shrg74104: 20 chunks from 313 turns
  tokens per chunk : min 68  median 6,047  max 6,144
  turns per chunk  : [16, 15, 1, 1, 1, 1, 1, 1, 8, 154, 77, 2, 2, 1, 1, 34, 1, 1, 1, 1]
  chunks from a split turn: 11

first 400 characters of the first chunk, as the model receives it:
------------------------------------------------------------------------------
Unidentified speaker: - HOW THE REPORT ON CARCINOGENS USES SCIENCE TO MEET ITS STATUTORY OBLIGATIONS, AND ITS IMPACT ON SMALL BUSINESS JOBS [House Hearing, 112 Congress] [From the U.S. Government Publishing Office]

HOW THE REPORT ON CARCINOGENS

USES SCIENCE TO MEET ITS STATUTORY

OBLIGATIONS, AND ITS IMPACT

ON SMALL BUSINESS JOBS =================================================================


## Step 9. Define one model call

Each call renders the system and user messages with the model's chat template, with the thinking mode off where offered, and samples near greedily. A completion that runs into its token cap is cut back to its last complete sentence. The map, collapse, and reduce steps differ only in their prompt and their cap: 320, 384, and 512 tokens.

In [ ]:
def render(tokenizer, system, user):
    """Apply the model's chat template, with the thinking mode off where the template offers it."""
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate(model, tokenizer, system, user, max_new_tokens):
    """One prompt, one completion, fresh context. Also returns whether the completion ran into the token cap."""
    prompt = render(tokenizer, system, user)
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    n_in = ids["input_ids"].shape[-1]
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id, **GENERATION)
    generated = out[0][n_in:]
    hit_cap = len(generated) >= max_new_tokens
    return tokenizer.decode(generated, skip_special_tokens=True).strip(), hit_cap


SENTENCE_END = tuple('.!?') + tuple(f'{p}{q}' for p in '.!?' for q in '"\')]')


def trim_to_sentence(text):
    """Cut a completion that hit the cap back to its last complete sentence."""
    text = text.rstrip()
    if not text or text.endswith(SENTENCE_END):
        return text
    cut = max(text.rfind(mark) for mark in '.!?')
    if cut <= 0:
        return text
    end = cut + 1
    while end < len(text) and text[end] in '"\')]':
        end += 1
    return text[:end].rstrip()


def summarize_chunk(chunk_text, model, tokenizer, max_new_tokens=320):
    """Map: one transcript segment to a summary of what was said in it."""
    text, hit_cap = generate(model, tokenizer, MAP_SYSTEM, MAP_USER.format(text=chunk_text), max_new_tokens)
    return trim_to_sentence(text) if hit_cap else text


def collapse_summaries(joined, model, tokenizer, max_new_tokens=384):
    """Collapse: a run of segment summaries to one shorter account."""
    text, hit_cap = generate(model, tokenizer, COLLAPSE_SYSTEM, COLLAPSE_USER.format(text=joined), max_new_tokens)
    return trim_to_sentence(text) if hit_cap else text


def reduce_summaries(joined, model, tokenizer, max_new_tokens=REDUCE_MAX_TOKENS):
    """Reduce: all segment summaries to the hearing's single summary, within the cap the downstream encoder needs."""
    text, hit_cap = generate(model, tokenizer, REDUCE_SYSTEM, REDUCE_USER.format(text=joined, target_words=REDUCE_TARGET_WORDS), max_new_tokens)
    return trim_to_sentence(text) if hit_cap else text

## Step 10. Define the map-reduce loop and the call log

Every chunk is summarized. Segment summaries that do not fit in one call are grouped within the budget and condensed, and that repeats until they fit. One reduce call then writes the hearing summary. Every call is logged as it happens, with a hash of its input and its full output, so the chain can be inspected afterwards.

In [ ]:
class CallLog:
    """One JSON line per model call, written as it happens: stage, level, position, a hash of the exact input, token counts, time, and the full output."""

    def __init__(self, path, document_id, model_name, prompt_version=None):
        self.path = Path(path) if path else None
        self.document_id = document_id
        self.model_name = model_name
        self.prompt_version = prompt_version or PROMPT_VERSION
        self.records = []
        if self.path:
            self.path.parent.mkdir(parents=True, exist_ok=True)

    def write(self, stage, level, index, n_inputs, prompt_text, output, tokenizer, seconds):
        record = {
            "document_id": self.document_id, "model": self.model_name, "prompt_version": self.prompt_version,
            "stage": stage, "level": level, "index": index, "n_inputs": n_inputs,
            "input_sha256": hashlib.sha256(prompt_text.encode("utf-8")).hexdigest(),
            "input_chars": len(prompt_text),
            "input_tokens": len(tokenizer.encode(prompt_text, add_special_tokens=False)),
            "output_tokens": len(tokenizer.encode(output, add_special_tokens=False)),
            "output_chars": len(output), "seconds": round(seconds, 2),
            "at": datetime.now(timezone.utc).isoformat(timespec="seconds"), "output": output,
        }
        self.records.append(record)
        if self.path:
            with self.path.open("a") as log_file:
                log_file.write(json.dumps(record) + "\n")
                log_file.flush()


def map_reduce(chunks, model, tokenizer, budget_tokens, max_levels=8, log=None, document_id="", model_name=""):
    """Map every chunk, collapse the summaries until they fit in one call, then reduce once."""
    if log is None:
        log = CallLog(None, document_id, model_name)

    level_summaries = []
    for i, chunk in enumerate(chunks):
        t0 = time.time()
        out = summarize_chunk(chunk, model, tokenizer)
        log.write("map", 0, i, 1, MAP_USER.format(text=chunk), out, tokenizer, time.time() - t0)
        level_summaries.append(out)
    levels = [list(level_summaries)]

    level = 0
    for _ in range(max_levels):  # a runaway guard, not a target; the level reached is reported per hearing
        joined = "\n\n".join(level_summaries)
        if len(tokenizer.encode(joined, add_special_tokens=False)) <= budget_tokens:
            break
        level += 1
        groups, current, current_tokens = [], [], 0  # regroup into batches that fit, then condense each
        for summary in level_summaries:
            n = len(tokenizer.encode(summary, add_special_tokens=False))
            if current and current_tokens + n > budget_tokens:
                groups.append(current)
                current, current_tokens = [], 0
            current.append(summary)
            current_tokens += n
        if current:
            groups.append(current)
        collapsed = []
        for i, group in enumerate(groups):
            text = "\n\n".join(group)
            t0 = time.time()
            out = collapse_summaries(text, model, tokenizer)
            log.write("collapse", level, i, len(group), COLLAPSE_USER.format(text=text), out, tokenizer, time.time() - t0)
            collapsed.append(out)
        level_summaries = collapsed
        levels.append(list(level_summaries))

    joined = "\n\n".join(level_summaries)
    t0 = time.time()
    final = reduce_summaries(joined, model, tokenizer)
    log.write("reduce", level + 1, 0, len(level_summaries), REDUCE_USER.format(text=joined, target_words=REDUCE_TARGET_WORDS), final, tokenizer, time.time() - t0)

    return {"final_summary": final, "final_tokens": len(tokenizer.encode(final, add_special_tokens=False)), "levels": levels,
            "n_levels": len(levels), "n_collapse_rounds": level, "n_calls": len(log.records), "n_chunks": len(chunks), "log": log}

## Step 11. Run map-reduce on both hearings

`levels` keeps every intermediate stage, so the input of the reduce step is visible rather than inferred.

In [6]:
results = []
for p, chunks in zip(picks, chunked):
    t0 = time.time()
    out = map_reduce([c["text"] for c in chunks], model, tokenizer, budget_tokens=CHUNK_TOKENS)
    out["packageId"] = p["packageId"]
    out["seconds"] = time.time() - t0
    results.append(out)
    print(f"{p['packageId']}: {out['n_chunks']} chunks, {out['n_levels']} level(s), {out['seconds']:.0f}s")

CHRG-112hhrg74059: 6 chunks, 1 level(s), 74s


CHRG-110shrg74104: 20 chunks, 1 level(s), 195s


## Step 12. Read the output

A summary that parses and has the right length can still be wrong. One segment summary and the final summary are printed in full next to the hearing's title.

In [7]:
for p, chunks, out in zip(picks, chunked, results):
    print("=" * 78)
    print(f"{p['packageId']}  ({p['year']}, {p['chamber']})")
    print(f"{p['title'][:110]}")
    print(f"{p['n_words']:,} words -> {out['n_chunks']} chunks -> {len(out['final_summary'].split()):,}-word / {out['final_tokens']}-token summary  [{out['seconds']:.0f}s]")
    print("=" * 78)
    print()
    print("--- one segment summary (map output, chunk 1) ---")
    print(out["levels"][0][0])
    print()
    print("--- final summary (reduce output) ---")
    print(out["final_summary"])
    print()

CHRG-112hhrg74059  (2012, HOUSE)
HOW THE REPORT ON CARCINOGENS USES SCIENCE TO MEET ITS STATUTORY OBLIGATIONS, AND ITS IMPACT ON SMALL BUSINESS
22,896 words -> 6 chunks -> 278-word / 392-token summary  [74s]

--- one segment summary (map output, chunk 1) ---
This segment covers a joint House hearing held on April 25, 2012, before the Subcommittee on Investigations and Oversight (Committee on Science, Space, and Technology) and the Subcommittee on Healthcare and Technology (Committee on Small Business), titled "How the Report on Carcinogens Uses Science to Meet Its Statutory Obligations and Its Impact on Small Business Jobs." The hearing focuses on the National Toxicology Program's (NTP) 12th Report on Carcinogens (RoC), its scientific methodology, regulatory implications, and economic impact on small businesses.

**Subjects and Positions:**
*   **Hon. Paul C. Broun (Chair, Science Committee):** Opened the hearing by expressing concern that while the RoC is not a direct regulation, it i

## Step 13. Check the run

These checks say the pipeline did what it claims, not that the summaries are good. Quality is judged against the C-SPAN abstracts in T1.3.

- Every chunk produced a segment summary.
- No output is empty at any level.
- The final summary is shorter than the transcript.
- The final summary is within the 512-token cap.
- Named speakers survive into the final summary.

In [8]:
for p, chunks, out in zip(picks, chunked, results):
    pid = p["packageId"]

    assert out["n_chunks"] == len(chunks), "chunk count disagrees with the input"
    assert len(out["levels"][0]) == len(chunks), "a chunk produced no map summary"
    print(f"{pid}: 1) every one of {len(chunks)} chunks produced a segment summary")

    empty = [i for i, s in enumerate(out["levels"][0]) if not s.strip()]
    assert not empty, f"segment summaries {empty} came back empty"
    assert out["final_summary"].strip(), "the final summary is empty"
    print(f"{pid}: 2) no empty output at any level")

    source_words = sum(len(c["text"].split()) for c in chunks)
    final_words = len(out["final_summary"].split())
    assert final_words < source_words, "the summary is longer than the transcript"
    print(f"{pid}: 3) {source_words:,} words in -> {final_words:,} out ({final_words / source_words:.2%})")

    assert out["final_tokens"] <= REDUCE_MAX_TOKENS, f"final summary is {out['final_tokens']} tokens, over the {REDUCE_MAX_TOKENS}-token cap"
    print(f"{pid}: 4) final summary {out['final_tokens']} tokens (cap {REDUCE_MAX_TOKENS})")

    named = {s for c in chunks for s in c["speakers"] if s != "Unidentified speaker"}
    kept = {s for s in named if s in out["final_summary"]}
    print(f"{pid}: 5) {len(kept)} of {len(named)} named speakers appear in the final summary")
    print()

print("Draft complete. Nothing was written to disk.")

CHRG-112hhrg74059: 1) every one of 6 chunks produced a segment summary
CHRG-112hhrg74059: 2) no empty output at any level
CHRG-112hhrg74059: 3) 23,426 words in -> 278 out (1.19%)
CHRG-112hhrg74059: 4) final summary 392 tokens (cap 512)
CHRG-112hhrg74059: 5) 9 of 14 named speakers appear in the final summary

CHRG-110shrg74104: 1) every one of 20 chunks produced a segment summary
CHRG-110shrg74104: 2) no empty output at any level
CHRG-110shrg74104: 3) 73,453 words in -> 321 out (0.44%)
CHRG-110shrg74104: 4) final summary 447 tokens (cap 512)
CHRG-110shrg74104: 5) 8 of 18 named speakers appear in the final summary

Draft complete. Nothing was written to disk.


## Step 14. Read the length test

Before the production run, three hearings were summarized with the final reduce target of 200 to 250 words. The point was to see where the token count lands under the 512-token cap. Two finished, at 250 and 264 words, 372 and 389 tokens; the third was not finished. This cell reads the saved logs and the shard file of that test.

In [1]:
test_folder = INTERIM_FOLDER / "lentest"
for log_file in sorted((test_folder / "status").glob("*.jsonl")):
    calls = pd.DataFrame([json.loads(line) for line in log_file.read_text().splitlines() if line.strip()])
    stages = calls["stage"].value_counts().to_dict()
    reduce_calls = calls.loc[calls["stage"] == "reduce"]
    if len(reduce_calls):
        words = len(reduce_calls["output"].iloc[0].split())
        print(f"{log_file.stem}: {len(calls)} calls {stages}; final summary {words} words, {int(reduce_calls['output_tokens'].iloc[0])} tokens")
    else:
        print(f"{log_file.stem}: {len(calls)} calls {stages}; not finished")

shard = pd.read_parquet(test_folder / "shards" / "summaries_00.parquet")
print()
print(shard[["packageId", "model", "prompt_version", "n_chunks", "n_calls", "summary_tokens", "seconds", "status"]].to_string(index=False))

CHRG-104hhrg37345: 12 calls {'map': 11, 'reduce': 1}; final summary 250 words, 372 tokens
CHRG-108shrg91310: 3 calls {'map': 3}; not finished
CHRG-109shrg25874: 9 calls {'map': 8, 'reduce': 1}; final summary 264 words, 389 tokens

        packageId           model        prompt_version  n_chunks  n_calls  summary_tokens  seconds status
CHRG-104hhrg37345 Qwen/Qwen3.5-4B hearing-mapreduce-1.0        11       12             372    150.9     ok
